In [43]:
import os
import tkinter as tk
from tkinter import filedialog, messagebox
from datetime import datetime

folder_path = ""

# --- Formats list with description ---
formats = [
    ("prefix_{number}", "Word at the beginning + number. Example: holiday_1.jpg"),
    ("{number}_prefix", "Number first + word at the end. Example: 1_holiday.jpg"),
    ("file_{number}", "Word 'file' + number. Example: file_1.jpg"),
    ("image_{number}", "Word 'image' + number. Example: image_1.jpg"),
    ("{number}_file", "Number + word 'file'. Example: 1_file.jpg"),
    ("prefix_{number}_{name}", "Word + number + original file name. Example: holiday_1_photo1.jpg"),
    ("{name}_{number}", "Original file name + number. Example: photo1_1.jpg"),
    ("{name}_{number}_final", "Original file name + number + word 'final'. Example: photo1_1_final.jpg"),
    ("YYYYMMDD_{number}", "Current date + number. Example: 20260314_1.jpg"),
    ("prefix_{number}_{ext}","Word + number + file extension. Example: holiday_1_jpg.jpg"),
]

format_names = [fmt for fmt, desc in formats]

# --- Functions ---
def select_folder():
    global folder_path
    folder_path = filedialog.askdirectory()
    folder_label.config(text=folder_path, fg="#1a73e8")  # blue text for selected folder

def rename_files():
    if folder_path == "":
        messagebox.showerror("Error", "Please select a folder")
        return

    choice_index = format_var.get()
    if choice_index == -1:
        messagebox.showerror("Error", "Please select a format")
        return

    user_prefix = prefix_entry.get().strip()
    start_number = number_entry.get().strip()
    start_number = int(start_number) if start_number.isdigit() else 1

    files = os.listdir(folder_path)
    if not files:
        messagebox.showinfo("Info", "No files found in the folder.")
        return

    today = datetime.today().strftime("%Y%m%d")

    for i, file in enumerate(files, start=start_number):
        name, ext = os.path.splitext(file)
        fmt = formats[choice_index][0]

        new_name = fmt
        if user_prefix and "prefix" in fmt:
            if fmt.startswith("prefix_"):
                fmt_no_prefix = fmt[len("prefix_"):]
                new_name = user_prefix + "_" + fmt_no_prefix
            elif fmt.endswith("_prefix"):
                fmt_no_prefix = fmt[:-len("_prefix")]
                new_name = fmt_no_prefix.replace("{number}", "{number}_" + user_prefix)

        new_name = new_name.replace("{number}", str(i))
        new_name = new_name.replace("{name}", name)
        new_name = new_name.replace("{ext}", ext.replace(".", ""))
        new_name = new_name.replace("YYYYMMDD", today)

        old_path = os.path.join(folder_path, file)
        new_path = os.path.join(folder_path, new_name + ext)
        os.rename(old_path, new_path)

    messagebox.showinfo("Success", "Files renamed successfully!")

# --- Tkinter GUI ---
root = tk.Tk()
root.title("Professional File Renamer")
root.geometry("650x500")
root.configure(bg="#f5f5f5")  # light gray background

# Folder selection
btn_folder = tk.Button(root, text="Select Folder", command=select_folder, bg="#1a73e8", fg="white", font=("Arial", 12, "bold"))
btn_folder.pack(pady=(20, 5))

folder_label = tk.Label(root, text="No folder selected", bg="#f5f5f5", fg="#555555", font=("Arial", 11))
folder_label.pack(pady=(0, 15))

# Custom prefix
prefix_label = tk.Label(root, text="Enter your custom prefix (optional, affects only 'prefix' formats):", bg="#f5f5f5", fg="#333333", font=("Arial", 11))
prefix_label.pack(pady=(5, 2))

prefix_entry = tk.Entry(root, font=("Arial", 12), width=30, bd=2, relief="groove")
prefix_entry.pack(pady=(0, 15))

# Starting number
number_label = tk.Label(root, text="Start numbering from (optional, default=1):", bg="#f5f5f5", fg="#333333", font=("Arial", 11))
number_label.pack(pady=(5, 2))

number_entry = tk.Entry(root, font=("Arial", 12), width=10, bd=2, relief="groove")
number_entry.pack(pady=(0, 15))

# Format selection
format_label = tk.Label(root, text="Choose a format:", bg="#f5f5f5", fg="#333333", font=("Arial", 11))
format_label.pack(pady=(5, 5))

format_var = tk.IntVar(value=-1)

def update_format_index(selection):
    try:
        format_var.set(format_names.index(selection))
    except ValueError:
        format_var.set(-1)

dropdown_var = tk.StringVar()
dropdown_var.set("Select a format")
dropdown = tk.OptionMenu(root, dropdown_var, *format_names, command=update_format_index)
dropdown.config(font=("Arial", 12), bg="white", fg="#1a73e8", width=35, relief="ridge")
dropdown["menu"].config(font=("Arial", 11), bg="white", fg="#1a73e8")
dropdown.pack(pady=(0, 15))

# Description of selected format
desc_label = tk.Label(root, text="", bg="#f5f5f5", fg="#777777", font=("Arial", 10, "italic"))
desc_label.pack(pady=(0, 15))

def update_desc(*args):
    index = format_var.get()
    if 0 <= index < len(formats):
        desc_label.config(text=formats[index][1])
    else:
        desc_label.config(text="")

dropdown_var.trace("w", update_desc)

# Rename button
rename_btn = tk.Button(root, text="Rename Files", command=rename_files, bg="#34a853", fg="white", font=("Arial", 13, "bold"), width=20)
rename_btn.pack(pady=(10, 20))

root.mainloop()